# Demo 1.2: Experiment Tracking for LLMs

Track LLM experiments systematically — parameters, metrics, costs, and tags.

In [ ]:
import sys; sys.path.insert(0, "..")

---
## Step 1: Environment Setup

In [ ]:
import os

import mlflow
import sys; sys.path.insert(0, "..")
from utils.clnt_utils import get_databricks_ai_gateway_client, get_openai_client, get_ai_gateway_model_names, is_databricks_ai_gateway_client
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Configure MLflow
mlflow.set_tracking_uri("databricks")

use_ai_gateway = is_databricks_ai_gateway_client()

# Verify which client to use
if use_ai_gateway:
    client = get_databricks_ai_gateway_client()
    model_name = get_ai_gateway_model_names()[0]
else:
    client = get_openai_client()
    model_name = "gpt-5.2"

# Verify OpenAI key
if not use_ai_gateway and not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found. Please check your .env file.")

# Enable autologging for OpenAI.
# This automatically creates MLflow Traces that capture:
#   - model, temperature, max_tokens (all API params as span attributes)
#   - full input messages and response content (span I/O)
#   - token counts: prompt_tokens, completion_tokens, total_tokens
#   - latency (via span start/end timestamps)
mlflow.openai.autolog()

print("✅ Environment configured successfully")
print(f"   MLflow Tracking URI: {mlflow.get_tracking_uri()}")
print(f"   Using model: {model_name}")
print("   Autolog: ENABLED")

---
## Step 2: Understanding Experiment Tracking

**Key concepts:** Parameters (config), Metrics (measurements), Artifacts (files), Tags (metadata), Traces (auto-captured LLM calls).

`mlflow.openai.autolog()` captures model params, token counts, latency, and I/O automatically. You only need manual `log_*` calls for cost estimates, semantic tags, and custom metrics.

---
## Step 3: First Tracked LLM Call

In [ ]:
# Create an experiment
experiment_name = "02-basic-llm-calls"
mlflow.set_experiment(experiment_name)

print(f"📊 Experiment: {experiment_name}")
print("   View in UI: https://dbc-baff2b7f-4402.cloud.databricks.com")

In [ ]:
# Make a tracked LLM call — autolog does the heavy lifting.
#
# What we do NOT need to log manually (autolog captures all of this):
#   - mlflow.log_param("model", ...)        -> span attribute
#   - mlflow.log_param("temperature", ...)   -> span attribute
#   - mlflow.log_param("max_tokens", ...)    -> span attribute
#   - mlflow.log_metric("latency_seconds")   -> span timestamps
#   - mlflow.log_metric("prompt_tokens")     -> mlflow.chat.tokenUsage
#   - mlflow.log_metric("completion_tokens") -> mlflow.chat.tokenUsage
#   - mlflow.log_metric("total_tokens")      -> mlflow.chat.tokenUsage
#   - mlflow.log_text(prompt, "prompt.txt")  -> span input
#   - mlflow.log_text(answer, "response.txt")-> span output

prompt = "Explain MLflow GenAI Platform in 3-4 sentences."

with mlflow.start_run(run_name="first-llm-tracked-call") as run:

    # The only explicit call: a semantic tag that autolog cannot infer.
    mlflow.set_tag("task", "explanation")

    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
        temperature=1.0,
        max_completion_tokens=1000
    )

    answer = response.choices[0].message.content

print(f"\n📝 Prompt: {prompt}")
print(f"\n🤖 Response: {answer}")
print(f"\n🔗 Run ID: {run.info.run_id}")
print(f"   View in UI: https://dbc-baff2b7f-4402.cloud.databricks.com/#/experiments/{run.info.experiment_id}/runs/{run.info.run_id}")

---
## Step 4: Comparing Multiple Configurations

In [ ]:
# Simplified helper — autolog captures params, tokens, latency, and I/O automatically.
def simple_llm_call(prompt, model=model_name, temperature=1.0, max_completion_tokens=1000, run_name=None):
    """
    Make an LLM call inside a nested run.
    autolog captures model params, token counts, latency, and full I/O as a Trace.
    The run exists only to group and name the experiment.
    """
    with mlflow.start_run(run_name=run_name, nested=True):
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=temperature,
            max_completion_tokens=max_completion_tokens
        )
        return response.choices[0].message.content

print("✅ Helper function defined!")

In [ ]:
# Create a new experiment for comparison
mlflow.set_experiment("02-temperature-comparison")

test_prompt = "Write a creative tagline for an AI observability with MLflow GenAI platform."
temperatures = [1.0, 1.5, 2.0]

print("🔬 Running temperature comparison...\n")

# A parent run groups all nested calls together in the UI.
with mlflow.start_run(run_name="temperature-sweep"):
    mlflow.set_tag("sweep_variable", "temperature")

    for temp in temperatures:
        print(f"  temperature={temp} ...")
        response = simple_llm_call(
            prompt=test_prompt,
            model=model_name,
            temperature=temp,
            max_completion_tokens=1000,
            run_name=f"temp_{temp}"
        )
        print(f"    -> {response}\n")

print("✅ Done. Compare traces side-by-side in the MLflow UI.")

---
## Step 5: Tracking Cost Estimates

MLflow 3.10+ records per-trace cost (input + output) automatically, making it easy to compare spend across models.

In [ ]:
def llm_call_with_cost(prompt, model=model_name, temperature=1.0, max_completion_tokens=1000, run_name=None):
    """
    Make an LLM call with cost tracking.
    autolog captures: model, temperature, token counts, latency, I/O.
    """
    with mlflow.start_run(run_name=run_name):
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=temperature,
            max_completion_tokens=max_completion_tokens
        )

        answer = response.choices[0].message.content

        return answer

print("✅ Cost-aware helper defined")
print("   autolog captures: model, temperature, token counts, latency, I/O")

In [ ]:
# Compare costs across different models
mlflow.set_experiment("03-model-cost-comparison")

prompt = "Summarize the benefits of experiment tracking in 3 bullet points."
models_to_test = ["jsd-gpt-5-2", "jsd-gpt-5-mini"] if use_ai_gateway else ["gpt-5-mini", "gpt-5.2"]

print("💰 Comparing costs across models...\n")

for model in models_to_test:
    print(f"Testing {model}...")
    response = llm_call_with_cost(
        prompt=prompt,
        model=model,
        temperature=1.0,
        max_completion_tokens=1000,
        run_name=f"model_{model}_run"
    )
    print(f"  Response: {response}...\n")

print("✅ Cost comparison complete! View in MLflow UI.")

---
## Step 6: Organizing with Tags and Metadata

Tags and structured configs add context autolog can't infer — team, stage, production candidacy.

In [ ]:
# Systematic experiment with rich metadata — tags and config artifacts.
mlflow.set_experiment("04-production-candidate-testing")

# Test configurations
open_configs = [
    {
        "name": "baseline",
        "model": "gpt-5-mini",
        "temperature": 1.0,
        "system_prompt": "You are a helpful assistant."
    },
    {
        "name": "creative",
        "model": "gpt-5.2",
        "temperature": 2.0,
        "system_prompt": "You are a creative writing assistant."
    },
]

# Databricks hosted foundational models if you want to test them
databricks_config = [
    {
        "name": "baseline",
        "model": "jsd-gpt-5-mini",
        "temperature": 1.0,
        "system_prompt": "You are a helpful assistant."
    },
    {
        "name": "creative",
        "model": "jsd-gpt-5.2",
        "temperature": 1.5,
        "system_prompt": "You are a creative writing assistant."
    },
]
model_configs = databricks_config if use_ai_gateway else open_configs
test_prompt = "Explain the concept of LLM temperature."

print("🏷️  Running experiments with semantic tags...\n")

for config in model_configs:
    with mlflow.start_run(run_name=config["name"]):

        # Make the call — autolog captures model, temperature, tokens, I/O, latency.
        response = client.chat.completions.create(
            model=config["model"],
            messages=[
                {"role": "system", "content": config["system_prompt"]},
                {"role": "user", "content": test_prompt}
            ],
            temperature=config["temperature"],
            max_completion_tokens=1000
        )

        # Log only what autolog cannot provide: semantic tags and structured config.
        mlflow.set_tags({
            "config_name": config["name"],
            "task": "explanation",
            "stage": "testing",
            "team": "ai-research",
            "version": "v1.0",
            "production_candidate": str(config["name"] == "baseline").lower(),
        })

        # Save full config as a structured artifact
        mlflow.log_dict(config, "config.json")

        print(f"  ✓ {config['name']} done")

print("\n✅ All runs completed! Filter by tag 'production_candidate=true' in the UI.")

---
## Step 7: Querying Experiments Programmatically

In [ ]:
from mlflow.tracking import MlflowClient

# Use the MlflowClient to query experiments and runs
mlflow_client = MlflowClient()

# Get experiment by name
experiment = mlflow_client.get_experiment_by_name("04-production-candidate-testing")

if experiment:
    print(f"📊 Experiment: {experiment.name}")
    print(f"   ID: {experiment.experiment_id}")

    # Search runs — sort by start_time since autolog stores latency on Traces, not run metrics.
    runs = mlflow_client.search_runs(
        experiment_ids=[experiment.experiment_id],
        order_by=["start_time DESC"],
        max_results=5
    )

    print(f"\n   Found {len(runs)} runs:\n" + "="*60)

    for run in runs:
        print(f"\n   Run: {run.info.run_name}")
        if run.data.params:
            print("   Parameters:")
            for key, value in run.data.params.items():
                print(f"      {key}: {value}")
        if run.data.metrics:
            print("   Metrics:")
            for key, value in run.data.metrics.items():
                print(f"      {key}: {value}")
        if run.data.tags.get("config_name"):
            print(f"   Tag config_name: {run.data.tags['config_name']}")
else:
    print("Experiment not found. Make sure you ran the production candidate testing section.")

In [ ]:
# Find production candidates using tag filters
if experiment:
    prod_runs = mlflow_client.search_runs(
        experiment_ids=[experiment.experiment_id],
        filter_string="tags.production_candidate = 'true'",
        max_results=5
    )

    print("🏆 Production Candidates:")
    for run in prod_runs:
        print(f"   Name: {run.info.run_name}")
        print(f"   Config: {run.data.tags.get('config_name', 'N/A')}")
        print(f"   Run ID: {run.info.run_id}")

---
## Reference: Search APIs

Use `mlflow_client.search_runs()` for run-level params/metrics/tags. Use `mlflow.search_traces()` for autolog data (model params, token counts, latency).